<a href="https://colab.research.google.com/github/y7q7/lerobot/blob/main/lerobot/training-act.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤗 x 🦾: Training ACT with LeRobot Notebook

Welcome to the **LeRobot ACT training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `ACT` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `ACT` policy for 100,000 steps typically takes **about 1.5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer.

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg, and installs the package in editable mode with train and dataset features.

In [1]:
!git clone https://github.com/huggingface/lerobot.git
!apt-get install ffmpeg
!cd lerobot && pip install -e ".[train, dataset]"

Cloning into 'lerobot'...
remote: Enumerating objects: 55841, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 55841 (delta 13), reused 9 (delta 9), pack-reused 55821 (from 2)
Receiving objects: 100% (55841/55841), 237.68 MiB | 27.34 MiB/s, done.
Resolving deltas: 100% (35380/35380), done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.
Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import lerobot

## Weights & Biases login (optional)
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging. This step is optional, you can skip it. If you want to use W&B remember to change `--wandb.enable` to true in the next section.

In [10]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: raojiawei10 (raojiawei10-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [11]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! Your code can use the GPU.")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")
else:
    print("CUDA is not available. Your code is running on CPU.")

CUDA is available! Your code can use the GPU.
Current CUDA device: 0
CUDA device name: Tesla T4


In [12]:
!nvidia-smi

Wed Sep 23 02:44:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## HF login

To upload your trained model to the hub you need to login with your Hugging Face account.
1. Run the cell below.
2. You will be asked to generate a token in the Hugging Face settings.
3. Select all checkboxes under Repositories when creating the token.
4. Paste the generated token into the command line below.

In [8]:
!hf auth login

? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: JUZR-14A5

    Waiting for authorization........
Token is valid.
The token `oauth-rjw0901` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-rjw0901`
Note: This token will be refreshed automatically when it expires.


## Start training ACT with LeRobot

This cell runs `lerobot-train` to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--policy.type=act`:  
   Specifies the policy configuration to use. `act` refers to [Action Chunking with Transformers](https://huggingface.co/docs/lerobot/act), which will automatically adapt to your dataset’s setup (e.g., number of motors and cameras).

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this. Set to `False` if you do not plan on using Weights & Biases.

7. `--batch_size=8`:  
   Increase it if you memmory allows it. It defines how many datapoints are processed at once.

8. `--steps=20000`:  
   Set for how many steps you want to train your model. 20 000 steps should work fine for a simple ACT policy.

In [13]:
!lerobot-train \
  --dataset.repo_id=lerobot/pusht \
  --policy.type=act \
  --output_dir=outputs/train/pusht_act_record0 \
  --job_name=pusht_act_training_job \
  --policy.device=cuda \
  --wandb.enable=True \
  --policy.repo_id=lerobot/pusht_policy0 \
  --batch_size=8 \
  --steps=20000

INFO 2026-09-23 02:45:32 ot_train.py:444 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'reshard_after_forward': True,
                          'wrap_modules': None},
                 'grad_scaler': {'backoff_factor': 0.5,
                                 'growth_factor': 2.0,
                                 'growth_interval': 2000,
                                 'init_scale': 65536.0},
                 'gradie

Sometimes after training, you may notice that the model underperforms and cannot solve the task properly. Sometimes this is due to poor data quality, but sometimes the model simply needs more training. To continue training from a previously trained model, use `--policy.pretrained_path=username/path_to_model` and paste the path to the model you trained previously here.